# 04 — Full Training

## Prerequisites
- Debug training (notebook 03) succeeded — loss decreased
- Data prepared (notebook 01) — `train.tsv` and `valid.tsv` exist
- Tokenizer trained (notebook 02) — `ur_sp.model` exists

## GPU Setup
This notebook requires a GPU. On Colab, go to Runtime → Change runtime type → GPU.
On Kaggle, enable GPU in the sidebar.

## Hyperparameters
- Batch size: 64
- Epochs: 12
- Learning rate: 3e-4
- Gradient clipping: 1.0
- Teacher forcing: 1.0

## Checkpoint Strategy
Only the best checkpoint (lowest validation loss) is saved.
This avoids saving checkpoints that overfit.

## Reproducibility
Seeds are set for Python, NumPy, PyTorch, and CUDA.
Note: exact bitwise reproducibility on GPU is not always guaranteed.

In [ ]:
# Colab/Kaggle setup
# !git clone https://github.com/YOUR_USERNAME/urdu-qgen-seq2seq.git
# %cd urdu-qgen-seq2seq
# !pip install -r requirements.txt

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
from torch.utils.data import DataLoader

from configs.config import *
from src.tokenizer.tokenizer_utils import UrduTokenizer
from src.data.dataset import QGenDataset, collate_fn
from src.model.encoder import Encoder
from src.model.decoder import Decoder
from src.model.seq2seq import Seq2Seq
from src.training.train import train_model
from src.training.utils import set_seed, get_device, count_parameters

In [ ]:
set_seed(SEED)
device = get_device()
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
tokenizer = UrduTokenizer(SP_MODEL_PATH)
vocab_size = tokenizer.vocab_size

train_dataset = QGenDataset(TRAIN_FILE, tokenizer)
valid_dataset = QGenDataset(VALID_FILE, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'Training:   {len(train_dataset):,} examples')
print(f'Validation: {len(valid_dataset):,} examples')

In [ ]:
enc_hidden = HIDDEN_SIZE * 2
encoder = Encoder(vocab_size, EMBEDDING_DIM, HIDDEN_SIZE, NUM_LAYERS, DROPOUT, PAD_ID)
decoder = Decoder(vocab_size, EMBEDDING_DIM, HIDDEN_SIZE, enc_hidden, NUM_LAYERS, DROPOUT, PAD_ID)
model = Seq2Seq(encoder, decoder).to(device)

print(f'Trainable parameters: {count_parameters(model):,}')

In [ ]:
config = {
    'vocab_size': vocab_size,
    'embedding_dim': EMBEDDING_DIM,
    'hidden_size': HIDDEN_SIZE,
    'num_layers': NUM_LAYERS,
    'dropout': DROPOUT,
    'bidirectional': BIDIRECTIONAL,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'epochs': EPOCHS,
    'seed': SEED,
}

logs = train_model(
    model, train_loader, valid_loader, device,
    epochs=EPOCHS, learning_rate=LEARNING_RATE,
    checkpoint_path=BEST_MODEL_PATH, config=config,
)

## Plot training/validation loss

In [ ]:
from src.visualization import plot_loss_curve

plot_loss_curve()